# Deploy Semantic Model Optimization Analytics

Run this notebook once in the target Fabric workspace. It creates or updates the Lakehouse, scanner notebook, ingestion pipeline, Direct Lake semantic model, and Power BI report. Re-running the same notebook upgrades items by name while preserving Lakehouse data.

After deployment, use **Load_SMO_Data** for all normal data collection runs.


In [ ]:
%pip install -q ms-fabric-cli PyYAML semantic-link-sempy


In [ ]:
# SOURCE SETTINGS — normally no changes are required.
repository_owner = "ZeonZheng"
repository_name = "fabric-semantic-model-optimization"
branch = "main"
source_archive_url_optional = ""
github_token_optional = ""  # Runtime-only fallback for a private repository; never save a real token.
_inlineInstallationEnabled = True


In [ ]:
import os
import sys
import tempfile
import zipfile
from io import BytesIO
from pathlib import Path

import requests

archive_url = (
    source_archive_url_optional.strip()
    or f"https://api.github.com/repos/{repository_owner}/{repository_name}/zipball/{branch}"
)
headers = {}
if github_token_optional.strip():
    headers["Authorization"] = f"Bearer {github_token_optional.strip()}"

response = requests.get(archive_url, headers=headers, timeout=120)
response.raise_for_status()

deployment_root = Path(tempfile.mkdtemp(prefix="smo-deploy-"))
with zipfile.ZipFile(BytesIO(response.content)) as archive:
    archive.extractall(deployment_root)

matches = list(deployment_root.rglob("config/deployment_config.yaml"))
if len(matches) != 1:
    raise RuntimeError(f"Expected one solution root, found {len(matches)}.")
repo_root = matches[0].parents[1]
sys.path.insert(0, str(repo_root))
print(f"Source loaded: {repository_owner}/{repository_name}@{branch}")


In [ ]:
from scripts.deploy_core import deploy_solution

deployment_result = deploy_solution(repo_root)
